## The Kubernetes object model

Open any Kubernetes manifest — Pod, Service, Deployment, ConfigMap, even a custom one — and you find the same five top-level keys. Memorise them once and you'll never feel lost in a new manifest.

```
apiVersion: <which API group + version this object belongs to>
kind:       <what kind of object this is>
metadata:   <name, namespace, labels, annotations — who this object is>
spec:       <desired state — what YOU want>
status:     <actual state — what the CLUSTER observes>
```

- **`apiVersion` names the API group.** Core objects (Pod, Service, ConfigMap, Namespace) use `v1`. Newer ones carry a group: `apps/v1` (Deployments), `batch/v1` (Jobs), `networking.k8s.io/v1` (Ingress, NetworkPolicy), `rbac.authorization.k8s.io/v1` (RBAC).
- **`kind`** — the object type, capitalised and singular: `Pod`, `Deployment`, `Service`.
- **`metadata`** — `name` (unique per namespace + kind), optional `namespace`, `labels`, `annotations`, plus system fields (`uid`, `creationTimestamp`).
- **`spec` is what you want.** *Your* contribution — `replicas: 3`, `image: nginx:alpine`.
- **`status` is what is.** *The cluster's* contribution — `phase: Running`, `podIP: 10.244.0.5`. You never write it; you read it.

### The reconcile loop — the whole game

```
   spec  ── desired (you wrote it)
     │
     ▼
 controller ── observe world · compare to spec · act to close the gap
     │
     ▼
  status ── actual (controller wrote it)
```

Every controller runs the same loop, forever: **observe, compare, act.** That's why we say "declarative" — you describe the destination, and a thousand little control loops drive the cluster toward it. `kubectl apply` updates `spec`; a controller notices; when it's done, `status` reflects reality. On our map this is enforced at the **api-server**, with **etcd** holding the object your spec became.